In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

# Util

In [ ]:
import os
from pathlib import Path
from typing import Optional
from typing import List

def get_file_path(file_path: str) -> str:
    """
    파일 경로를 절대 경로로 변환하는 함수
    
    Args:
        file_path: 상대 경로 또는 절대 경로
    """
    # 파일 경로 확인 및 절대 경로로 변환
    file_path = Path(file_path)
    if not file_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        file_path = project_root / file_path
    
    if not file_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {file_path}")
    
    return str(file_path)


def table_to_markdown(table: List[List]) -> str:
    """
    표 데이터를 마크다운 테이블 형식으로 변환하는 헬퍼 함수
    
    Args:
        table: 2차원 리스트 형태의 표 데이터
    
    Returns:
        마크다운 테이블 문자열
    """
    if not table or len(table) == 0:
        return ""
    
    # 빈 셀을 빈 문자열로 변환
    def clean_cell(cell):
        if cell is None:
            return ""
        return str(cell).strip()
    
    # 표 데이터 정리
    cleaned_table = [[clean_cell(cell) for cell in row] for row in table]
    
    # 최대 컬럼 수 확인
    max_cols = max(len(row) for row in cleaned_table) if cleaned_table else 0
    
    # 모든 행을 동일한 컬럼 수로 맞춤
    normalized_table = []
    for row in cleaned_table:
        normalized_row = row + [""] * (max_cols - len(row))
        normalized_table.append(normalized_row)
    
    if not normalized_table:
        return ""
    
    markdown_lines = []
    
    # 헤더 행 (첫 번째 행)
    header = normalized_table[0]
    markdown_lines.append("| " + " | ".join(header) + " |")
    
    # 구분선
    markdown_lines.append("| " + " | ".join(["---"] * len(header)) + " |")
    
    # 데이터 행들
    for row in normalized_table[1:]:
        markdown_lines.append("| " + " | ".join(row) + " |")
    
    return "\n".join(markdown_lines)

# LLM 모델 생성

In [ ]:
# 문서가 변액일임펀드 설정/해지 지시서인지 확인하는 LLM 노드 생성

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
# LLM 모델 정의

LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

def create_llm_model():
    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=LLM_TEMPERATURE,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.9로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    return llm

# pdfplumber

In [ ]:
def extract_text_from_pdf_with_pdfplumber(pdf_path: str, password: str = None) -> str:
    """
    pdfplumber를 사용하여 PDF 파일을 마크다운 형식으로 변환하는 함수
    
    pdfplumber는 PDF 파일을 텍스트 데이터로 추출하는 라이브러리로, 표, 이미지, 레이아웃 등을 잘 보존합니다.
    암호화된 PDF와 암호화되지 않은 PDF 모두 처리할 수 있습니다.
    """
    try:
        import pdfplumber
    except ImportError:
        raise ImportError(
            "PDF를 처리하기 위해 pdfplumber가 필요합니다.\n"
            "설치 명령: pip install pdfplumber"
        )
    
    markdown_parts = []
    
    try:
        pdf_path = get_file_path(pdf_path)
        # password가 있으면 암호화된 PDF로 처리, 없으면 암호화되지 않은 PDF로 처리
        pdf_kwargs = {"password": password} if password else {}
        
        with pdfplumber.open(pdf_path, **pdf_kwargs) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                page_content = []
                
                # 표 추출 (표가 있으면 먼저 표를 추출)
                tables = page.extract_tables()
                if tables:
                    for table_idx, table in enumerate(tables):
                        if table:
                            markdown_table = table_to_markdown(table)
                            if markdown_table:
                                page_content.append(markdown_table)
                                page_content.append("")  # 표 다음에 빈 줄 추가
                
                # 텍스트 추출
                text = page.extract_text()
                if text:
                    page_content.append(text)
                
                if page_content:
                    markdown_parts.append("\n".join(page_content))
        
        return "\n\n".join(markdown_parts) if markdown_parts else ""
        
    except Exception as e:
        # 암호화 관련 오류인지 확인
        error_msg = str(e).lower()
        if 'password' in error_msg or 'encrypted' in error_msg or 'incorrect password' in error_msg:
            raise ValueError(f"PDF 암호가 올바르지 않거나 암호화된 PDF를 읽을 수 없습니다: {e}")
        raise

# Docling

In [ ]:
from typing import Optional
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableFormerMode
from docling.datamodel.accelerator_options import AcceleratorOptions

# ✅ 표 구조 복원에 유리한 권장 백엔드 (기본값이기도 함) :contentReference[oaicite:5]{index=5}
from docling.backend.docling_parse_v4_backend import DoclingParseV4DocumentBackend

try:
    from docling.document_converter import PdfBackendOptions
except ImportError:
    from docling.datamodel.base_models import PdfBackendOptions

from docling_core.types.doc.document import ContentLayer  # :contentReference[oaicite:3]{index=3}

def extract_text_from_pdf_with_docling3(pdf_path: str, password: str = None) -> str:
    pdf_path = get_file_path(pdf_path)

    def _run(do_cell_matching: bool) -> str:
        pipeline = PdfPipelineOptions(
            do_ocr=False,
            do_table_structure=True,
            do_picture_classification=False,
            do_picture_description=False,
            generate_page_images=True,
            images_scale=2.0,  # 레이아웃/테이블 크롭 품질에 도움될 수 있음 :contentReference[oaicite:6]{index=6}
        )

        # ✅ 표 구조 품질 우선
        pipeline.table_structure_options.mode = TableFormerMode.ACCURATE  # :contentReference[oaicite:7]{index=7}
        pipeline.table_structure_options.do_cell_matching = do_cell_matching  # :contentReference[oaicite:8]{index=8}

        # ✅ 표 구조 목적이면 force_backend_text는 끄는 쪽이 안전
        pipeline.force_backend_text = True  # :contentReference[oaicite:9]{index=9}

        pipeline.accelerator_options = AcceleratorOptions(device="cpu")

        backend_opts = PdfBackendOptions(password=password) if password else None

        converter = DocumentConverter(
            allowed_formats=[InputFormat.PDF],
            format_options={
                InputFormat.PDF: PdfFormatOption(
                    pipeline_options=pipeline,
                    backend=DoclingParseV4DocumentBackend,
                    backend_options=backend_opts,
                )
            },
        )

        result = converter.convert(pdf_path)
        # md = result.document.export_to_markdown()
        md = result.document.export_to_markdown(
            included_content_layers={ContentLayer.BODY},
            page_break_placeholder="\n\n<!-- pagebreak -->\n\n",                 # (선택) 페이지 경계 표시 :contentReference[oaicite:5]{index=5}
        )

        # “테이블이 전혀 안 잡혔는지” 빠른 휴리스틱 (파이프 문자 기반)
        # 필요하면 여기서 result.document에서 TableItem 개수를 세는 방식으로 더 정확히 판단 가능 :contentReference[oaicite:10]{index=10}
        return md

    # 1차: 기본(셀 매칭 True)
    md = _run(do_cell_matching=True)
    if "|---" in md or "| ---" in md:
        print("do_cell_matching=True")
        return md

    # 2차: 셀 매칭 False (borderless/매칭 실패 케이스에 유리) :contentReference[oaicite:11]{index=11}
    md2 = _run(do_cell_matching=False)
    print("do_cell_matching=False")
    return md2


# document load

In [ ]:
# Document Loader - 파일 형식에 따라 적절한 함수 호출

from pathlib import Path
from typing import Union

def load_document(file_path: str, password: str = None, pdf_lib: str = 'pdfplumber') -> str:
    """
    파일 형식에 따라 적절한 텍스트 추출 함수를 호출하여 텍스트를 반환하는 통합 함수
    
    지원 형식:
    - PDF: .pdf 파일 (암호화된 PDF 지원)
    - Excel: .xlsx, .xls 파일
    
    Args:
        file_path: 문서 파일 경로 (상대 경로 또는 절대 경로)
        password: PDF 파일이 암호화된 경우 비밀번호 (선택사항)
    
    Returns:
        추출된 텍스트 문자열
        - PDF: 전체 텍스트
        - Excel: 모든 시트의 텍스트를 합친 문자열
    
    Raises:
        FileNotFoundError: 파일을 찾을 수 없을 때
        ValueError: 지원하지 않는 파일 형식일 때 또는 PDF 암호가 틀렸을 때
        ImportError: 필요한 라이브러리가 설치되지 않았을 때
    """
    file_path_obj = Path(file_path)
    file_ext = file_path_obj.suffix.lower()
    
    # 파일 형식에 따라 적절한 함수 호출
    if file_ext == '.pdf':
        # PDF 파일 처리 (암호 전달)
        if pdf_lib == 'pdfplumber':
            return extract_text_from_pdf_with_pdfplumber(file_path, password)
        elif pdf_lib == 'docling':
            return extract_text_from_pdf_with_docling3(file_path, password)
    
    # elif file_ext in ['.xlsx', '.xls']:
    #     # Excel 파일 처리 - 모든 시트의 텍스트를 하나의 문자열로 반환
    #     return get_all_sheets_text(file_path)
    
    else:
        raise ValueError(
            f"지원하지 않는 파일 형식입니다: {file_ext}\n"
            f"지원 형식: .pdf, .xlsx, .xls"
        )

# LLM 문서 정리

In [ ]:
# system 프롬프트
_SYSTEM_PROMPT = """당신은 자산운용사에서 해외거래체결 확인을 담당하는 오퍼레이터 입니다.
    당신의 역할은 시스템에 자동 피딩된 해외거래체결내역 정보와 브로커가 메일로 보내온 해외거래체결내역 확인서를 비교하여, 시스템에 오입력된 정보가 있으면 수정하고 미입력된 정보는 등록하는 역할입니다.
"""

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown(document_text: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
    아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber 라이브러리를 사용하여 추출한 data입니다.
    추출 data는 table을 markdown 형식으로 변환하여 추출한 data와 텍스트 형식으로 추출한 data로 구성되어 있습니다.
    지침에 따라 추출 data를 정리하세요.
    
    ### 변액일임펀드 설정/해지 지시서 내용 ###
    {document_text}

    ** 반드시 지켜야 할 중요 지침 **
    1. 전체 내용을 분석하세요.
    2. 전체 내용을 markdown 형식으로 수정하세요.
    3. markdown table 코드를 오류가 없는 정상적인 코드로 수정하세요.    
    4. 추측과 예상 또는 설명 등의 첨언은 하지 말고 추출 data 내용만 출력하세요.
    5. 중복되는 내용은 제거하세요.
    6. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    7. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    8. 정리 결과에서 종목명과 코드가 정확한지 확인하고 오류가 있으면 수정하세요.
    9. 정리 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown2(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
    아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.
    추출 data는 table을 markdown 형식으로 변환하여 추출한 data와 텍스트 형식으로 추출한 data로 구성되어 있습니다.
    지침에 따라 추출 data를 정리하세요.
    
    ### 변액일임펀드 설정/해지 지시서 내용 - docling 라이브러리 사용 ###
    {document_text_docling}    

    ### 변액일임펀드 설정/해지 지시서 내용 - pdfplumber 라이브러리 사용 ###
    {document_text_pdfplumber}

    ** 반드시 지켜야 할 중요 지침 **
    1. docling 추출 data를 기반으로 전체 내용을 분석하세요.
    2. 각 단어와 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.
    3. 전체 내용을 markdown 형식으로 수정하세요.
    4. markdown table 코드를 오류가 없는 정상적인 코드로 수정하세요.
    5. 중복되는 내용은 제거하세요.
    6. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    7. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    8. 정리 결과의 테이블에서 컬럼 데이터가 인접한 컬럼에 중복되어 작성되어 있는지 확인하세요. 중복되어 작성되어 있으면 제거하세요.
    9. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.
    10. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.
    11. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.
    12. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
    13. 종목명(stock name, security name) 지침
        13.1. 종목명(stock name, security name, name)은 시스템 표준이므로 단어의 철자를 추가하거나 변경하지말고 그대로 유지하세요.
        13.2. **종목명(stock name, security name, name)은 시스템 표준이므로 축약된 단어가 사용될 수 있습니다. 이를 복원하면 시스템에서 오류가 발생할 수 있으므로 이를 복원하지 마세요.**
        13.3. **단어의 띄어쓰기, 공백, 줄바꿈 오류만 수정하세요.**
        13.4. 종목명(stock name, security name, name)의 의미를 분석하고 맥락을 통해 단어의 띄어쓰기 및 공백 오류가 있는지 확인하고 오류가 있으면 수정하세요.
        13.5. 종목명(stock name, security name, name)의 의미를 분석하여 맥락을 통해 단어가 중간에서 줄바꿈으로 잘려진 사실이 확인되면 수정하세요.           
    14. 원문을 번역하지 말고 원문 그대로 출력하세요.
    15. 추측과 예상 또는 설명 등의 첨언은 하지 말고 추출 data 내용만 출력하세요.
    16. 정리 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

def convert_document_text_to_markdown3(document_text_docling: str, document_text_pdfplumber: str):

    # 메시지 객체 생성
    system_msg = SystemMessage(_SYSTEM_PROMPT)
    human_msg = HumanMessage(f"""
    아래는 브로커가 보내온 주식 해외거래체결내역 확인서 PDF 파일에서 pdfplumber와 docling 라이브러리를 사용하여 추출한 data입니다.
    아래의 주식 해외거래체결내역 확인서 추출 data를 시스템에 자동 피딩된 정보와 비교 자료로 사용할 수 있도록 정리하세요.
    지침에 따라 정리한 내용을 markdown 형식으로 작성하세요.
    
    ### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
    {document_text_docling}    

    ### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
    {document_text_pdfplumber}

    ** 반드시 지켜야 할 중요 지침 **
    1. docling 추출 data를 기반으로 전체 내용을 분석하세요.
    2. 통합 또는 요약하지 말고, 거래 단위로 data를 정리하세요.
    3. 각 단어와 코드, 숫자 데이터들을 pdfplumber 추출 data와 비교하여 보다 정확한 data를 선택하세요.
    4. 정리 결과에서 누락된 데이터가 있는지 확인하세요. 누락된 데이터가 있으면 추가하세요.
    5. 정리 결과에서 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    6. PDF에서 추출한 data의 특성상 인접한 컬럼의 데이터가 중복되거나, 인접한 컬럼으로 병합되는 오류가 발생할 수 있습니다. 정리 결과의 테이블에서 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 pdfplumber 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.
    7. 정리 결과에서 단어 또는 숫자를 임의로 제거하지 말고 있는 그대로 출력하세요.
    8. 정리 결과에서 단어 또는 숫자를 임의로 요약하지 말고 있는 그대로 출력하세요.
    9. 정리 결과에서 단어 또는 숫자를 임의로 새로 작성하지 말고 있는 그대로 출력하세요.
    10. PDF에서 추출한 data의 특성상 공백, 띄어쓰기, 줄바꿈 오류가 발생할 수 있습니다. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
    11. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
    12. 종목명(stock name, security name, name)에서 약어를 사용하는지 판단하여 약어를 유지하세요.  
    13. 원문을 번역하지 말고 원문 그대로 출력하세요.
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# LLM 문서 검증

In [ ]:
def validation_markdown_document_with_llm(document_text_docling: str, document_text_pdfplumber: str, document_markdown: str):

    # 메시지 객체 생성
    system_msg = SystemMessage("""당신은 자산운용사에서 해외거래체결 확인을 담당하는 오퍼레이터 입니다.
    당신의 역할은 시스템에 자동 피딩된 해외거래체결내역 정보와 브로커가 메일로 보내온 해외거래체결내역 확인서를 비교하여, 시스템에 오입력된 정보가 있으면 수정하고 미입력된 정보는 등록하는 역할입니다.
    """)
    human_msg = HumanMessage(f"""
    markdown으로 정리한 주식 해외거래체결내역 확인서 내용을 검수하세요.

    pdfplumber와 docling 라이브러리를 사용하여 주식 해외거래체결내역 확인서 PDF 파일에서 추출한 원문 data를 참고하여 아래의 지침에 따라 검수하세요.

    ### 주식 해외거래체결내역 확인서 markdown 정리 내용 - 검수 대상 ###
    {document_markdown}
    
    ### 주식 해외거래체결내역 확인서 PDF 파일 내용 - docling 라이브러리 사용 ###
    {document_text_docling}    

    ### 주식 해외거래체결내역 확인서 PDF 파일 내용 - pdfplumber 라이브러리 사용 ###
    {document_text_pdfplumber}

    ** 반드시 지켜야 할 중요 지침 **
    1. 누락된 데이터가 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 누락된 데이터가 있으면 추가하세요.
    2. 금액과 합계 데이터가 있는지 확인하세요. 합계가 정확한지 검증하세요.
    3. 테이블의 데이터가 인접한 컬럼에 중복되거나 병합되어 작성되어 있는지 PDF 파일 내용 추출 data와 비교하여 확인하세요. 중복 또는 병합되어 있으면 수정하세요.    
    4. 단어의 의미를 분석하고 맥락을 통해 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
    5. 종목명(stock name, security name, name)에서 공백, 띄어쓰기, 줄바꿈 오류가 자주 발생합니다. 종목명에서 공백, 띄어쓰기, 줄바꿈 오류가 존재하는지 확인하고 오류가 있으면 단어의 의미와 맥락에 맞도록 수정하세요.
    6. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 단어의 철자가 다르거나 단어가 추가된 경우 원문을 유지하도록 수정하세요.
    7. 종목명(stock name, security name, name)을 PDF 파일 내용 추출 data와 비교하여 약어를 풀로 표기한 경우 약어로 수정하세요.  
    8. 검수 결과의 전체 내용을 검증하고 오류가 있으면 수정하세요.
    9. 설명 및 요약 등의 첨언은 하지 말고 시스템 입력에 필요한 주식 해외거래체결내역 확인 정보만 출력하세요.    
    """)

    # 채팅 모델과 함께 사용
    messages = [system_msg, human_msg]

    llm = create_llm_model()
    response = llm.invoke(messages)  # AIMessage 반환

    return response

# 테스트

In [ ]:
from IPython.display import Markdown, display

def display_markdown(response):
    # LLM 응답을 마크다운 형식으로 보기 좋게 표시
    if 'response' in locals():
        display(Markdown(response.content))
        
        # 추가 정보 (토큰 사용량 등)를 표시
        if hasattr(response, 'response_metadata') and response.response_metadata:
            metadata = response.response_metadata
            if 'token_usage' in metadata:
                print("\n---")
                print("**토큰 사용량:**")
                print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
                print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
                print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
    else:
        print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")

In [293]:
# text 추출

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/대신.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/메리츠.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/미래.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/삼성.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/신한.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/유진.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/키움.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/하나.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/한국.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/DB.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/JP Morgan.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/KB.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/NH.pdf"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/SB(Bernstein).pdf"

_password = None
_password = '345678'

document_text_pdfplumber = load_document(_document_file_path, _password, pdf_lib='pdfplumber')
print(document_text_pdfplumber)

Email Notice of Execution message Bernstein Institutional Services, LLC
245 Park Avenue
This is a NEW Notice of Execution New York, NY 10167
Tel:(917) 344 8575
Attention of: SAMSUNG ASSET MANAGEMENT CO.,LTD.
Company: SAMSUNG ASSET MANAGEMENT CO.,LTD.
Email/Fax Address: globalop@samsung.com
From Email: AMEROps@bernsteinsg.com
Date: Aug 26, 2025
Trade Reference: 0000000216674524
Traded Time: 20250826 09:30:00.913
Order Type: LMT
Venue: MLT ***
Security: S&P GLOBAL INC
Ticker: SPGI
SEDOL Code: BYV2325
ISIN Code: US78409V1044
Account: 7011253890
BOK EQ ESG PASSIVE SAMSUNG(SAM_783011)
We have SOLD for you as AGENT
249.000 shares in S&P GLOBAL INC at a price of USD 550.3271
Traded on Aug 26, 2025. Settlement due on Aug 27, 2025.
Charges
Gross Consideration: USD 137,031.45
Exec Commission: USD 41.11
Research Commission: USD 0.00
Total Commission: USD 0.00
Local Fee: USD 0.00
Local Tax: USD 0.00
Stamp Duty: USD 0.00
Net Consideration: USD 136,990.34
Exchange Rate: 1.000000000
We will receive v

In [294]:
document_text_docling = load_document(_document_file_path, _password, pdf_lib='docling')
print(document_text_docling)

2026-01-07 20:22:55,605 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-01-07 20:22:55,608 - INFO - Going to convert document batch...
2026-01-07 20:22:55,608 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 3a85fbd3fac1234bc9da7957f69a8ff6
2026-01-07 20:22:55,609 - INFO - Accelerator device: 'cpu'
2026-01-07 20:22:56,394 - INFO - Accelerator device: 'cpu'
2026-01-07 20:22:56,881 - INFO - Processing document SB(Bernstein).pdf
2026-01-07 20:22:57,852 - INFO - Finished converting document SB(Bernstein).pdf in 2.25 sec.
2026-01-07 20:22:57,854 - WARNING - Parameter `strict_text` has been deprecated and will be ignored.


do_cell_matching=True
## Email Notice of Execution message

## This is a NEW Notice of Execution

Attention of:

SAMSUNG ASSET MANAGEMENT CO.,LTD.

Company:

SAMSUNG ASSET MANAGEMENT CO.,LTD.

Email/Fax Address:

globalop@samsung.com

From Email:

AMEROps@bernsteinsg.com

Date:

Aug 26, 2025

Trade Reference:

0000000216674524

Traded Time:

20250826 09:30:00.913

Order Type:

LMT

Venue:

MLT ***

Security:

S&amp;P GLOBAL INC

Ticker:

SPGI

SEDOL Code:

BYV2325

ISIN Code:

US78409V1044

Account:

7011253890

BOK EQ ESG PASSIVE SAMSUNG(SAM\_783011)

## We have SOLD for you as AGENT

Bernstein Institutional Services, LLC

245 Park Avenue

New York, NY 10167

Tel:(917) 344 8575

249.000 shares in S&amp;P GLOBAL INC at a price of USD 550.3271

## Traded on Aug 26, 2025 . Settlement due on

Aug 27, 2025.

## Charges

| Gross Consideration:   | USD   | 137,031.45   |
|------------------------|-------|--------------|
| Exec Commission:       | USD   | 41.11        |
| Research Commission:

In [295]:
res_markdown = convert_document_text_to_markdown3(document_text_docling, document_text_pdfplumber)
display_markdown(res_markdown)

2026-01-07 20:23:05,985 - INFO - HTTP Request: POST http://localhost:3900/v1/chat/completions "HTTP/1.1 200 OK"


```markdown
# 주식 해외거래체결내역 확인서 (정리본)

## 기본 정보
- **Attention of:** SAMSUNG ASSET MANAGEMENT CO.,LTD.  
- **Company:** SAMSUNG ASSET MANAGEMENT CO.,LTD.  
- **Email/Fax Address:** globalop@samsung.com  
- **From Email:** AMEROps@bernsteinsg.com  
- **Date:** Aug 26, 2025  
- **Trade Reference:** 0000000216674524  
- **Traded Time:** 20250826 09:30:00.913  
- **Order Type:** LMT  
- **Venue:** MLT ***  
- **Security:** S&P GLOBAL INC  
- **Ticker:** SPGI  
- **SEDOL Code:** BYV2325  
- **ISIN Code:** US78409V1044  
- **Account:** 7011253890  
- **Account Name:** BOK EQ ESG PASSIVE SAMSUNG(SAM_783011)  

## 거래 내역
- **Direction:** SOLD  
- **Agent:** Bernstein Institutional Services, LLC  
- **Quantity:** 249.000 shares  
- **Security:** S&P GLOBAL INC  
- **Price:** USD 550.3271  

## 결제 정보
- **Trade Date:** Aug 26, 2025  
- **Settlement Date:** Aug 27, 2025  

## 금액 내역 (Charges)
| 항목 | 통화 | 금액 |
|------|------|------|
| Gross Consideration | USD | 137,031.45 |
| Exec Commission | USD | 41.11 |
| Research Commission | USD | 0.00 |
| Total Commission | USD | 0.00 |
| Local Fee | USD | 0.00 |
| Local Tax | USD | 0.00 |
| Stamp Duty | USD | 0.00 |
| Net Consideration | USD | 136,990.34 |
| Exchange Rate | - | 1.000000000 |

> **합계 검증:**  
> Gross Consideration (137,031.45) - Exec Commission (41.11) = Net Consideration (136,990.34) → **정확함**

## 결제 지시 정보
### Our Delivery Instructions:
- **Clearing system:** DTC  
- **Account:** 0286  

### Client Delivery Instructions:
- **SettlementAgent:** SBVQ  
- **Agent:** 29796  
- **Clearing:** 00997  

### Settlement Consideration:
- **Amount:** 136,990.34  
- **Currency:** USD (US Dollar)  

## 기타 정보
- **Settlement Method:** We will receive versus payment  
- **Contact for Queries:**  
  - Email: AMEROps@bernsteinsg.com  
  - Tel: (917) 344 8575  
- **Trade Reference for Inquiry:** 0000000216674524  
- **Objection Period:** Within one Business Day of dispatch  
- **Business Day Definition:** A day which is not a Saturday or a Sunday and on which banks are open for business.  
- **Note:** This is not a regulatory confirmation and is provided for informational purposes only. A more detailed confirmation compliant with SEC Rule 10b-10 will follow.  
- **Do Not Reply:** Please do not reply to this mail.  
```


---
**토큰 사용량:**
- 입력 토큰: 2191
- 출력 토큰: 843
- 총 토큰: 3034


# 재 검증

In [296]:
valid_markdown = validation_markdown_document_with_llm(document_text_docling, document_text_pdfplumber, res_markdown)
display_markdown(valid_markdown)

2026-01-07 20:23:13,941 - INFO - HTTP Request: POST http://localhost:3900/v1/chat/completions "HTTP/1.1 200 OK"


```markdown
# 주식 해외거래체결내역 확인서 (정리본)

## 기본 정보
- **Attention of:** SAMSUNG ASSET MANAGEMENT CO.,LTD.  
- **Company:** SAMSUNG ASSET MANAGEMENT CO.,LTD.  
- **Email/Fax Address:** globalop@samsung.com  
- **From Email:** AMEROps@bernsteinsg.com  
- **Date:** Aug 26, 2025  
- **Trade Reference:** 0000000216674524  
- **Traded Time:** 20250826 09:30:00.913  
- **Order Type:** LMT  
- **Venue:** MLT ***  
- **Security:** S&P GLOBAL INC  
- **Ticker:** SPGI  
- **SEDOL Code:** BYV2325  
- **ISIN Code:** US78409V1044  
- **Account:** 7011253890  
- **Account Name:** BOK EQ ESG PASSIVE SAMSUNG(SAM_783011)  

## 거래 내역
- **Direction:** SOLD  
- **Agent:** Bernstein Institutional Services, LLC  
- **Quantity:** 249.000 shares  
- **Security:** S&P GLOBAL INC  
- **Price:** USD 550.3271  

## 결제 정보
- **Trade Date:** Aug 26, 2025  
- **Settlement Date:** Aug 27, 2025  

## 금액 내역 (Charges)
| 항목 | 통화 | 금액 |
|------|------|------|
| Gross Consideration | USD | 137,031.45 |
| Exec Commission | USD | 41.11 |
| Research Commission | USD | 0.00 |
| Total Commission | USD | 0.00 |
| Local Fee | USD | 0.00 |
| Local Tax | USD | 0.00 |
| Stamp Duty | USD | 0.00 |
| Net Consideration | USD | 136,990.34 |
| Exchange Rate | - | 1.000000000 |

> **합계 검증:**  
> Gross Consideration (137,031.45) - Exec Commission (41.11) = Net Consideration (136,990.34) → **정확함**

## 결제 지시 정보
### Our Delivery Instructions:
- **Clearing system:** DTC  
- **Account:** 0286  

### Client Delivery Instructions:
- **SettlementAgent:** SBVQ  
- **Agent:** 29796  
- **Clearing:** 00997  

### Settlement Consideration:
- **Amount:** 136,990.34  
- **Currency:** USD (US Dollar)  

## 기타 정보
- **Settlement Method:** We will receive versus payment  
- **Contact for Queries:**  
  - Email: AMEROps@bernsteinsg.com  
  - Tel: (917) 344 8575  
- **Trade Reference for Inquiry:** 0000000216674524  
- **Objection Period:** Within one Business Day of dispatch  
- **Business Day Definition:** A day which is not a Saturday or a Sunday and on which banks are open for business.  
- **Note:** This is not a regulatory confirmation and is provided for informational purposes only. A more detailed confirmation compliant with SEC Rule 10b-10 will follow.  
- **Do Not Reply:** Please do not reply to this mail.  
```


---
**토큰 사용량:**
- 입력 토큰: 3204
- 출력 토큰: 843
- 총 토큰: 4047
